In [1]:
1+1

2

In [12]:
from langchain_core.documents import Document
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

### RAG PIPELINE

In [21]:
### read all the documents in the directory
def read_all_documents(pdf_directory):
    """Read all the documents from the pdf"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"{len(pdf_files)}to process")
    for pdf_file in pdf_files:
        print(f"processing{pdf_file.name}")
        try:
            Loader = PyPDFLoader(str(pdf_file))
            documents = Loader.load()
            ## add source to the metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error processing  {e}")
            
    print(f"the number of {len(all_documents)}")
    return all_documents
all_the_documents = read_all_documents("../data")
        


3to process
processing1-4 Deep learnng.pdf
loaded 5 pages
processing5-8 Deep LEarning.pdf
loaded 6 pages
processingKeerthi_Vardhan_Genpact_DataScientist.pdf
loaded 2 pages
the number of 13


In [22]:
all_the_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\1-4 Deep learnng.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': '1-4 Deep learnng.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\1-4 Deep learnng.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2', 'source_file': '1-4 Deep learnng.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\1-4 Deep learnng.pdf', 'total_pages': 5, 'page': 2, 'page_label': '3', 'source_file': '1-4 Deep learnng.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\1-4 Deep learnng.pdf', 'total_pages': 5, 'page': 3, 'page_label': '4', 'source_file': '1-4 Deep learnng.pdf', 'file_type': 'p

In [24]:
type(all_the_documents[0])

langchain_core.documents.base.Document

In [25]:
## TextSplitter to split the documents into smalle chunks
def split_documnets(documents,chunk_size = 1000,chunk_overlap=200):
    """split doduments into smaller chunks for better RAG performance"""
    split_doc = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    splitter_text = split_doc.split_documents(documents)
    print(f"split {len(documents)} into {len(splitter_text)} chunks")
    
    ## Show example of a chunk
    if splitter_text:
        print(f"Example of a chunk: {splitter_text[0].page_content[:200]}...")
        print(f"Metadata of the chunk: {splitter_text[0].metadata}")
        
    return splitter_text

In [26]:
chunk = split_documnets(all_the_documents)
chunk

split 13 into 6 chunks
Example of a chunk: KEERTHI VARDHAN NAIDU NETTEM
GitHub | LinkedIn | nk.vardhannaidu@gmail.com | +91-9949902603
PROFESSIONAL SUMMARY
Aspiring Data Scientist with hands-on experience in Artificial Intelligence, Machine Le...
Metadata of the chunk: {'producer': '', 'creator': 'WPS Docs', 'creationdate': '2026-06-14T10:13:35+05:30', 'author': 'Un-named', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-06-14T10:13:35+05:30', 'sourcemodified': "D:20260614101335+05'30'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Keerthi_Vardhan_Genpact_DataScientist.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Keerthi_Vardhan_Genpact_DataScientist.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': '', 'creator': 'WPS Docs', 'creationdate': '2026-06-14T10:13:35+05:30', 'author': 'Un-named', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-06-14T10:13:35+05:30', 'sourcemodified': "D:20260614101335+05'30'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Keerthi_Vardhan_Genpact_DataScientist.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Keerthi_Vardhan_Genpact_DataScientist.pdf', 'file_type': 'pdf'}, page_content='KEERTHI VARDHAN NAIDU NETTEM\nGitHub | LinkedIn | nk.vardhannaidu@gmail.com | +91-9949902603\nPROFESSIONAL SUMMARY\nAspiring Data Scientist with hands-on experience in Artificial Intelligence, Machine Learning, Predictive Analytics, and\nAdvanced Analytics. Skilled in developing and deploying end-to-end AI/ML solutions, performing exploratory data analysis,\nand delivering actionable recommendations from complex datasets. Experienced in collaborating with cross-functional\nte